In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# ==========================================
# 1. DATA COLLECTION (Yahoo Finance)
# ==========================================
# Fetch S&P 500 ETF (SPY) data
ticker = "SPY"
df = yf.download(ticker, start="2019-01-01", end="2024-01-01")

# Handle MultiIndex columns if returned by recent yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Calculate Daily Returns and Intraday Volatility (High - Low Percentage)
df["Return"] = df["Close"].pct_change()
df["Volatility"] = (df["High"] - df["Low"]) / df["Low"]

# Define variables:
# X_t = Volume Ratio (Volume normalized relative to 20-day moving average)
# Y_{t+1} = Next-day Volatility
df["Vol_MA20"] = df["Volume"].rolling(window=20).mean()
df["X_Volume_Ratio"] = df["Volume"] / df["Vol_MA20"]
df["Y_Next_Volatility"] = df["Volatility"].shift(-1)

# Drop missing values resulting from rolling calculations and shifting
data = df[["X_Volume_Ratio", "Y_Next_Volatility"]].dropna()

# ==========================================
# 2. PROVING STATISTICAL DEPENDENCE
# P[X=x, Y=y] != P[X=x] * P[Y=y]
# ==========================================
# Convert continuous variables to discrete bins to test joint probabilities
data["X_Bin"] = pd.qcut(data["X_Volume_Ratio"], q=2, labels=["Low_Vol", "High_Vol"])
data["Y_Bin"] = pd.qcut(
    data["Y_Next_Volatility"], q=2, labels=["Low_Vola", "High_Vola"]
)

contingency_table = pd.crosstab(
    data["X_Bin"], data["Y_Bin"], normalize=True
)
print("--- Joint Probability Distribution P[X, Y] ---")
print(contingency_table)

# Chi-Square Test for Statistical Independence
chi2, p_val, dof, ex = stats.chi2_contingency(
    pd.crosstab(data["X_Bin"], data["Y_Bin"])
)
print(f"\nChi-Square Test P-Value: {p_val:.6f}")
if p_val < 0.05:
    print(
        "RESULT: Reject Independence! X and Y are statistically associated."
    )

# Spearman Rank Correlation
corr, corr_p = stats.spearmanr(
    data["X_Volume_Ratio"], data["Y_Next_Volatility"]
)
print(f"Spearman Correlation: {corr:.4f} (p-value: {corr_p:.6f})")

# ==========================================
# 3. ASSOCIATIONAL PREDICTION: E[Y | X = x]
# ==========================================
X = data[["X_Volume_Ratio"]]
y = data["Y_Next_Volatility"]

# Train/Test split without shuffling to respect time-series ordering
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Fit a Random Forest Regressor (ML Predictive Tool)
ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X_train, y_train)

y_pred = ml_model.predict(X_test)
print(f"\nPredictive Model R^2 Score on Unseen Test Data: {r2_score(y_test, y_pred):.4f}")

# Compare Conditional Expectations E[Y | X]
low_volume_subset = data[data["X_Volume_Ratio"] < 0.8]["Y_Next_Volatility"]
high_volume_subset = data[data["X_Volume_Ratio"] > 1.2]["Y_Next_Volatility"]

print("\n--- Conditional Expectations E[Y | X = x] ---")
print(f"E[Next Volatility | Low Volume Today]  = {low_volume_subset.mean():.4%}")
print(f"E[Next Volatility | High Volume Today] = {high_volume_subset.mean():.4%}")

C:\Users\LocalUserName\AppData\Local\Temp\ipykernel_22104\1109502013.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed


--- Joint Probability Distribution P[X, Y] ---
Y_Bin     Low_Vola  High_Vola
X_Bin                        
Low_Vol   0.289984   0.210016
High_Vol  0.210016   0.289984

Chi-Square Test P-Value: 0.000000
RESULT: Reject Independence! X and Y are statistically associated.
Spearman Correlation: 0.2479 (p-value: 0.000000)

Predictive Model R^2 Score on Unseen Test Data: -3.0071

--- Conditional Expectations E[Y | X = x] ---
E[Next Volatility | Low Volume Today]  = 1.1457%
E[Next Volatility | High Volume Today] = 1.7813%
